# 06 汇总判定

前五个 Notebook 已把每条声明写入 `reports/claims/<ID>.json`。本 Notebook：

1. 运行 Tier C 的登记项（它们会立即返回 `NOT_REPRODUCIBLE` 并附带原因）；
2. 汇总全部 39 条声明，统计 verdict；
3. 生成 `verdicts.json`、`report.md`、`report.html`。

> 判定不是“论文好/坏”的评分。它回答的是：**这句话在给定资源下能否被独立复核，复核结果是否与原文一致。**


In [ ]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


## Tier C：容器内无法复现的条目

这些条目没有实现上的失败，只是需要论文级算力或未公开的数据/代码。显式记录它们，
是为了让报告不出现 PENDING，也避免把“没做”误读为“通过”。


In [ ]:
c_results = run_many(["C01", "C02", "C03", "C04", "C05"])
for cid, res in c_results.items():
    print(f"{cid}: {res.verdict.value} — {res.notes_zh[:80]}...")


## 全部声明一览


In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from ebba.claims.model import load_registry

registry = load_registry()
rows = []
for cid in registry:
    path = config.REPORTS_DIR / "claims" / f"{cid}.json"
    if not path.exists():
        rows.append({"id": cid, "tier": registry[cid]["tier"], "verdict": "PENDING",
                     "title_zh": registry[cid]["title_zh"]})
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows.append({"id": cid, "tier": payload["tier"], "verdict": payload["verdict"],
                 "title_zh": payload["title_zh"]})

nbutil.show_table([r["id"] for r in rows])
print("rows:", len(rows), "| PENDING:", sum(r["verdict"] == "PENDING" for r in rows))


## 判定统计与图


In [ ]:
counts = Counter(r["verdict"] for r in rows)
order = ["CONFIRMED", "PARTIAL", "CONTRADICTED", "NOT_REPRODUCIBLE", "ERROR", "PENDING"]
for key in order:
    print(f"{key:<18} {counts.get(key, 0)}")

colors = {"CONFIRMED": "#1b7f3a", "PARTIAL": "#9a6700", "CONTRADICTED": "#b42318",
          "NOT_REPRODUCIBLE": "#5f6b7a", "ERROR": "#7a1fa2", "PENDING": "#c9d1d9"}
fig, ax = plt.subplots(figsize=(6.5, 3.2))
keys = [k for k in order if counts.get(k, 0)]
ax.bar(keys, [counts[k] for k in keys], color=[colors[k] for k in keys])
ax.set_ylabel("claims")
ax.set_title("Verdict distribution")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## Tier 分布


In [ ]:
by_tier = {}
for r in rows:
    by_tier.setdefault(r["tier"], Counter())[r["verdict"]] += 1
for tier in sorted(by_tier):
    print(tier, dict(by_tier[tier]))


## 被否定的声明

下面逐条列出 `CONTRADICTED` 的实测要点。完整实测值在 `reports/report.html` 的折叠块中。


In [ ]:
for r in rows:
    if r["verdict"] != "CONTRADICTED":
        continue
    payload = json.loads((config.REPORTS_DIR / "claims" / f'{r["id"]}.json').read_text(encoding="utf-8"))
    print(f'{r["id"]}  {r["title_zh"]}\n   {payload["notes_zh"]}\n')


## 部分成立与不可复现

`PARTIAL` 有两种来源：证明前提不完整（A07、A19），以及缩比实证只支持方向（所有 B 声明）。
`NOT_REPRODUCIBLE` 是 Tier C，附有所需资源说明。


In [ ]:
for r in rows:
    if r["verdict"] in {"PARTIAL", "NOT_REPRODUCIBLE"}:
        print(f'{r["id"]:>4}  {r["verdict"]:<18} {r["title_zh"]}')


## 运行耗时与环境

`reports/run_metadata.json` 记录每个 Notebook 的状态与耗时；`reports/verdicts.json` 是机器可读的最终产物。


In [ ]:
meta_path = config.REPORTS_DIR / "run_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print("device:", meta.get("device"), "| fast:", meta.get("fast"),
          "| total:", meta.get("total_seconds"), "s")
    for rec in meta["notebooks"]:
        print(f'  {rec["name"]:<18} {rec["status"]:<8} {rec.get("seconds", "-")}s')
else:
    print("run_metadata.json 不存在（直接打开 Notebook 时正常）")


## 结论怎么引用

1. **Tier A 的 `CONTRADICTED`** 是可直接复核的事实：附录 B 数值、表格算术、BA 下界反例、
   ASR 门槛反例、GTSRB 笔误、Table 3/4 过度声明、官方代码与论文协议不符。引用论文时应注明这些例外。
2. **Tier B 的 `PARTIAL`** 只说明方法机制在缩比设定下按预期工作，**不能**写成“复现了论文的 95% PDR”。
3. **Tier C 的 `NOT_REPRODUCIBLE`** 是范围声明，不是负面结论。
4. 任何 `ERROR` 或 `PENDING` 都应视为交付缺陷；`verify` 的退出码会反映执行故障。

最终报告：`reports/report.html`（自包含，可离线打开）与 `reports/verdicts.json`。
